# 2026-10-21 Distance and absolute magnitude

## Overview

+ Make a color-magnitude diagram from your photometry
+ Get distances to your stars from Gaia
+ Correct the instrumental magnitudes for distance

## Color-magnitude diagram...maybe an HR diagram?

In [ ]:
from pathlib import Path
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

from astroquery.gaia import Gaia

from stellarphot import PhotometryData
from ipyautoui.custom import FileChooser

%matplotlib inline
import matplotlib.pyplot as plt

#### Choose your photometry file (should be called `photometry.ecsv')

In [ ]:
p = Path("/calibrated/color-images-2026")
start = str(p) if p.exists() else "."
chooser = FileChooser(filter_pattern=["*.ecsv"], path=start)
chooser

In [ ]:
your_mag_table = chooser.value

In [ ]:
mag_table = PhotometryData.read(your_mag_table)

In [ ]:
B = mag_table[mag_table['passband'] == 'B']
V = mag_table[mag_table['passband'] == 'V']

In [ ]:
mag_col = 'mag_inst'
# fig, resid = plt.subplots(1, 1, figsize=(10, 10))

# resid.plot(B['color_cat'], B['color_cat'] - (B[mag_col] - V[mag_col]), '.', label='catalog colors', alpha=0.4)
# #plt.ylim(16.5, 10)
# resid.set_xlabel('Catalog B-V')
# resid.set_ylabel('Diff between catalog and calibrated B-V')
# resid.set_title(your_mag_table)
# resid.grid()

In [ ]:
bad_V_sel = False
bad_B_sel = False

if set(V['star_id']) ^ set(B['star_id']):
    bad_V = [bad_id for bad_id in (set(V['star_id']) - set(B['star_id']))]
    bad_B = [bad_id for bad_id in (set(B['star_id']) - set(V['star_id']))]
    for bad in bad_V:
        bad_V_sel |= V['star_id'] == bad
    for bad in bad_B:
        bad_B_sel |= B['star_id'] == bad
    if bad_V_sel is not False:
        V = V[~bad_V_sel]
    if bad_B_sel is not False:
        B = B[~bad_B_sel]

In [ ]:
good_color = np.abs(B[mag_col]) > -1  # B['color_cat'] - (B['mag_inst_cal'] - V['mag_inst_cal'])) < 0.1

In [ ]:
fig, cmd = plt.subplots(1, 1, figsize=(5, 5))

cmd.plot(B[mag_col][good_color] - V[mag_col][good_color], V[mag_col][good_color], '.')
cmd.set_ylim(*cmd.get_ylim()[::-1])
cmd.set_ylabel('V')
cmd.set_xlabel('B - V')
cmd.set_xlim(-0.5, 2.5)
cmd.set_xlabel("Color (B-V instrumental)")
cmd.set_ylabel("Magnitude (V instrumental")
cmd.set_title("Color-magnitude diagram (???)")
cmd.grid()


## Correcting for distance

In [ ]:
our_coord_V = SkyCoord(V['ra'], V['dec'])
center = SkyCoord(ra=our_coord_V.ra.mean(), dec=our_coord_V.dec.mean())
search_radius = our_coord_V.separation(center).max() + 1 * u.arcmin
center, search_radius.to(u.arcmin)

In [ ]:
Gaia.ROW_LIMIT = -1
j = Gaia.cone_search_async(center, radius=search_radius)
r = j.get_results()
gaia_cs = SkyCoord(ra=r['ra'], dec=r['dec'], unit='degree')
len(r)

In [ ]:
idxg, d2dg, _ = our_coord_V.match_to_catalog_sky(gaia_cs)
parallax = np.asarray(r['parallax'].filled(np.nan))[idxg]
great = (d2dg < 1 * u.arcsec) & (parallax > 0)
distance = 1000 / parallax  # parsecs, parallax is in milliarcsec
great.sum()

In [ ]:
abs_mag_V = np.full(len(V), np.nan)
abs_mag_V[great] = V[mag_col][great] - 5 * (np.log10(distance[great]) - 1)
V['abs_mag_inst'] = abs_mag_V

In [ ]:
good = good_color & great

fig, (app, absol) = plt.subplots(1, 2, figsize=(10, 5))

app.plot(B[mag_col][good] - V[mag_col][good], V[mag_col][good], '.')
app.set_ylim(*app.get_ylim()[::-1])
app.set_xlim(-0.5, 2.5)
app.set_xlabel("Color (B-V instrumental)")
app.set_ylabel("Apparent magnitude (V instrumental)")
app.grid()

absol.plot(B[mag_col][good] - V[mag_col][good], V['abs_mag_inst'][good], '.')
absol.set_ylim(*absol.get_ylim()[::-1])
absol.set_xlim(-0.5, 2.5)
absol.set_xlabel("Color (B-V instrumental)")
absol.set_ylabel("Absolute magnitude (V instrumental)")
absol.grid()

## Compare to a real HR diagram

In [ ]:
atlas_hr = plt.imread('media/atlas-universe-hr.png')

fig, (yours, atlas) = plt.subplots(1, 2, figsize=(14, 6))

yours.plot(B[mag_col][good] - V[mag_col][good], V['abs_mag_inst'][good], '.')
yours.set_ylim(*yours.get_ylim()[::-1])
yours.set_xlim(-0.5, 2.5)
yours.set_xlabel("Color (B-V instrumental)")
yours.set_ylabel("Absolute magnitude (V instrumental)")
yours.set_title("Your stars")
yours.grid()

atlas.imshow(atlas_hr)
atlas.set_axis_off()
atlas.set_title("Atlas of the Universe")